# 04 — Scratch CNN · Deliverable 2

The **Step 3.1 baseline CNN, recipe unchanged**: 128 px, batch 32, four conv blocks, augmentation as layers,
balanced class weights, early stopping on `val_loss` with best-weight restore. Nothing here is tuned — this is
the reference the transfer backbones are measured against, and the paired model for the Grad-CAM comparison.

What is different from the Step 3.1 notebook is only the *plumbing*:

- the split is **loaded** from `00_split/` on Drive and its fingerprint printed — never rebuilt;
- the trained model is written to `02_models/scratch_cnn.keras` **immediately after `fit`, before any evaluation**,
  so a runtime wipe cannot cost the run again;
- a **skip-guard** loads that file and skips training if it is already there;
- metrics go to `03_results/`, the confusion matrix to `04_figures/`.

> **Runtime ▸ Change runtime type ▸ GPU (T4)** first. Full run ≈ 25 min. Set `SMOKE_TEST = True` in the config
> cell for a ~2-minute dry run that writes only `_SMOKE`-suffixed artifacts.

Expect macro-F1 ≈ 0.979, plausibly 0.96–0.98 — GPU training is not bit-deterministic and this project has
~0.02 of documented session noise. **Record whatever this run produces; do not reuse the old number.**

## Colab setup — run this first

Mounts Drive (where the split, the model and the results live) and loads the Kaggle credentials from Colab
Secrets with the retry we use everywhere, because the Secrets service is intermittently slow to answer.

In [ ]:
# Colab setup: Kaggle credentials from Secrets (with retry) + mount Drive. Harmless when run locally.
import os, time

ON_COLAB = False
try:
    from google.colab import userdata, drive
    ON_COLAB = True
except ModuleNotFoundError:
    print("Not on Colab - using local ~/.kaggle/kaggle.json and a local folder in place of Drive.")

if ON_COLAB:
    ok = False
    for attempt in range(1, 4):
        try:
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
            print(f"Kaggle credentials loaded from Colab Secrets (attempt {attempt}).")
            ok = True
            break
        except Exception as e:
            print(f"  attempt {attempt}: Secrets not ready ({type(e).__name__}); retrying in 3s...")
            time.sleep(3)
    if not ok:
        print("")
        print("Colab Secrets did not respond. Fixes, in order:")
        print("  1) RE-RUN this cell - the timeout is almost always transient.")
        print("  2) Left sidebar -> key icon -> confirm KAGGLE_USERNAME and KAGGLE_KEY exist,")
        print("     each with 'Notebook access' toggled ON, then re-run.")
        print("  3) Still failing? Set os.environ['KAGGLE_USERNAME'/'KAGGLE_KEY'] manually here.")
    drive.mount("/content/drive")

Kaggle credentials loaded from Colab Secrets (attempt 1).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
drive = Path("/content/drive/MyDrive")
print("Drive mounted        :", drive.exists())
pr = drive / "plant_recognition"
print("plant_recognition on Drive:", pr.exists())
if pr.exists():
    for p in sorted(pr.rglob("*")):
        print("   ", p.relative_to(pr), "/" if p.is_dir() else "")
print("\nsplit_train.csv on Drive :", list(pr.rglob("split_train.csv")) if pr.exists() else "-")
print("split_train.csv locally  :", list(Path.home().rglob("split_train.csv")))


Drive mounted        : True
plant_recognition on Drive: True
    baseline_cnn.keras 
    deliverable2 /
    deliverable2/02_models /
    deliverable2/03_results /
    deliverable2/04_figures /
    step3_4_interpretability /
    step3_4_interpretability/attention_concentration.png 
    step3_4_interpretability/attention_on_leaf.csv 
    step3_4_interpretability/attention_summary.csv 
    step3_4_interpretability/baseline_cnn.keras 
    step3_4_interpretability/checkpoint_verification.csv 
    step3_4_interpretability/confusion_pairs.csv 
    step3_4_interpretability/gradcam_failures.png 
    step3_4_interpretability/gradcam_panel.png 
    step3_4_interpretability/run_config.json 
    step3_baseline /
    step3_baseline/split_test.csv 
    step3_baseline/split_train.csv 
    step3_baseline/split_val.csv 

split_train.csv on Drive : [PosixPath('/content/drive/MyDrive/plant_recognition/step3_baseline/split_train.csv')]
split_train.csv locally  : []


In [ ]:
# One-off: promote the frozen split into the deliverable2 layout and fingerprint it.
from pathlib import Path
import hashlib, shutil, pandas as pd

SRC = Path("/content/drive/MyDrive/plant_recognition/step3_baseline")
D2  = Path("/content/drive/MyDrive/plant_recognition/deliverable2")
DST = D2 / "00_split"
for d in ("00_split", "01_notebooks", "05_report"):
    (D2 / d).mkdir(parents=True, exist_ok=True)

for s in ("train", "val", "test"):
    shutil.copy2(SRC / f"split_{s}.csv", DST / f"split_{s}.csv")   # copy, not move

dfs   = {s: pd.read_csv(DST / f"split_{s}.csv") for s in ("train", "val", "test")}
sizes = tuple(len(dfs[s]) for s in ("train", "val", "test"))
sets  = {s: set(dfs[s]["filepath"]) for s in dfs}
assert sizes == (38013, 8146, 8146), f"sizes {sizes} - this is NOT the canonical split"
assert len(sorted(dfs["train"]["label"].unique())) == 38
assert sets["train"].isdisjoint(sets["val"]) and sets["train"].isdisjoint(sets["test"]) \
       and sets["val"].isdisjoint(sets["test"]), "LEAKAGE"
print("checks pass:", sizes, "| 38 classes | disjoint by file path")

def md5(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

dig = {s: md5(DST / f"split_{s}.csv") for s in ("train", "val", "test")}
SPLIT_ID = hashlib.md5("".join(dig[s] for s in ("train","val","test")).encode()).hexdigest()[:12]
lines  = ["# canonical split fingerprint - plant_recognition / deliverable2",
          "# seed=42  split=(0.70, 0.15, 0.15)  stratified, file-level",
          f"# promoted from step3_baseline/ on {pd.Timestamp.now():%Y-%m-%d %H:%M}, not rebuilt"]
lines += [f"split_{s}.csv  rows={sizes[i]:>6,}  md5={dig[s]}" for i, s in enumerate(("train","val","test"))]
lines += [f"SPLIT_ID = {SPLIT_ID}"]
(DST / "fingerprint.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
print("\nfirst train path:", dfs["train"]["filepath"].iloc[0])

checks pass: (38013, 8146, 8146) | 38 classes | disjoint by file path
# canonical split fingerprint - plant_recognition / deliverable2
# seed=42  split=(0.70, 0.15, 0.15)  stratified, file-level
# promoted from step3_baseline/ on 2026-07-28 09:29, not rebuilt
split_train.csv  rows=38,013  md5=7d083aaec519a8d188f5b2349ea27b06
split_val.csv  rows= 8,146  md5=611d00bb5835e052b177d58de357d0af
split_test.csv  rows= 8,146  md5=6ee6c1b7fd1df3942357a5460188848c
SPLIT_ID = 9e33ec57c1ec

first train path: /root/plant_recognition/plantvillage/plantvillage dataset/color/Grape___Esca_(Black_Measles)/2850cff3-e262-49b1-af09-051e76e19fa8___FAM_B.Msls 1336.JPG


## 1. Configuration

**Why run it:** one block holds the whole recipe, so "unchanged recipe" is something you can read off the screen
rather than take on trust.

**How it fits:** `IMG_SIZE`, `BATCH_SIZE`, `EPOCHS` and `SEED` are copied verbatim from
`Step3_1_PlantVillage_Baseline_Models`. The smoke run appends `_SMOKE` to every artifact name, so a dry run can
never overwrite a real model or a real results CSV — and the skip-guard can never mistake one for the other.

In [ ]:
# ---- the one config block: everything below reads from here ----
from pathlib import Path
import sys, subprocess, hashlib, random, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (f1_score, recall_score, precision_score, accuracy_score,
                             confusion_matrix, classification_report)

sns.set_theme()
SEED = 42
random.seed(SEED); np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)          # seeds python / numpy / tensorflow together

# ---- the Step 3.1 recipe, unchanged ----
IMG_SIZE   = 128
BATCH_SIZE = 32
EPOCHS     = 20                               # EarlyStopping normally stops around epoch 12-16

# ---- run switches ----
SMOKE_TEST      = False                       # True -> ~2-min dry run, writes only *_SMOKE artifacts
SMOKE_PER_CLASS = 60
FORCE_RETRAIN   = False                       # True -> retrain and overwrite an existing scratch_cnn.keras

# ---- deliverable2 tree on Drive ----
_drive     = Path("/content/drive/MyDrive")
DRIVE_ROOT = _drive if _drive.exists() else Path.home()          # local fallback off Colab
D2         = DRIVE_ROOT / "plant_recognition" / "deliverable2"
SPLIT_DIR  = D2 / "00_split"
MODELS     = D2 / "02_models"
RESULTS    = D2 / "03_results"
FIGURES    = D2 / "04_figures"
for d in (MODELS, RESULTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
assert SPLIT_DIR.exists(), f"{SPLIT_DIR} does not exist - run 00_setup_and_split.ipynb first."

SPLIT_CSV   = {s: SPLIT_DIR / f"split_{s}.csv" for s in ("train", "val", "test")}
FINGERPRINT = SPLIT_DIR / "fingerprint.txt"

# ---- artifact names (a smoke run must not touch the real ones) ----
TAG        = "_SMOKE" if SMOKE_TEST else ""
MODEL_PATH = MODELS  / f"scratch_cnn{TAG}.keras"
HIST_PATH  = RESULTS / f"scratch_cnn_history{TAG}.json"
RES_CSV    = RESULTS / f"scratch_cnn_results{TAG}.csv"
PC_CSV     = RESULTS / f"scratch_cnn_per_class_recall{TAG}.csv"
LC_PNG     = FIGURES / f"scratch_cnn_learning_curves{TAG}.png"
CM_PNG     = FIGURES / f"scratch_cnn_confusion_matrix{TAG}.png"

# ---- raw images: local runtime disk, not Drive (Drive is far too slow to train from) ----
PV_DIR   = Path.home() / "plant_recognition" / "plantvillage"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff"}

gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow {tf.__version__} | GPU: {'YES - ' + gpus[0].name if gpus else 'NO (Runtime > Change runtime type > T4)'}")
print(f"recipe : IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, SEED={SEED}")
print(f"run    : SMOKE_TEST={SMOKE_TEST}, FORCE_RETRAIN={FORCE_RETRAIN}")
print(f"model  : {MODEL_PATH}")
if SMOKE_TEST:
    print("\n*** SMOKE_TEST is ON - all artifacts carry the _SMOKE suffix; the real ones are untouched. ***")

AssertionError: /content/drive/MyDrive/plant_recognition/deliverable2/00_split does not exist - run 00_setup_and_split.ipynb first.

## 2. Load the frozen split and print the fingerprint

**Why run it:** every comparable number in the report depends on this being the *same* split. The CSVs are read
from Drive and the `SPLIT_ID` re-derived from their bytes, so a mismatch fails here rather than surfacing later
as an unexplainable score.

**How it fits:** this notebook never calls `train_test_split`. If the printed `SPLIT_ID` differs from the one in
`00_split/fingerprint.txt`, the assertion stops the run — **do not rebuild the split to make it pass.**

The test CSV is read for its row count and its MD5 only. It is not turned into a dataset and never scored: it
stays sealed for the single final evaluation.

In [ ]:
# Load the canonical split from Drive (never rebuilt) and re-derive its fingerprint
for s, p in SPLIT_CSV.items():
    assert p.exists(), f"missing {p} - run 00_setup_and_split.ipynb first"

dfs = {s: pd.read_csv(SPLIT_CSV[s]) for s in ("train", "val", "test")}

def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

digests  = {s: md5(SPLIT_CSV[s]) for s in ("train", "val", "test")}
SPLIT_ID = hashlib.md5("".join(digests[s] for s in ("train", "val", "test")).encode()).hexdigest()[:12]

recorded = None
if FINGERPRINT.exists():
    for line in FINGERPRINT.read_text().splitlines():
        if line.startswith("SPLIT_ID"):
            recorded = line.split("=")[-1].strip()

print("SPLIT_ID =", SPLIT_ID, "  |  fingerprint.txt says:", recorded if recorded else "(not found)")
assert recorded is None or recorded == SPLIT_ID, (
    "SPLIT_ID MISMATCH - STOP. This is not the split every other number was produced on.")

# ---- canonical label order and the hard checks ----
class_names = sorted(dfs["train"]["label"].unique())
name2idx    = {n: i for i, n in enumerate(class_names)}
n_classes   = len(class_names)
sizes       = (len(dfs["train"]), len(dfs["val"]), len(dfs["test"]))
assert n_classes == 38, f"expected 38 classes, found {n_classes}"
assert sizes == (38_013, 8_146, 8_146), f"split sizes {sizes} != canonical (38013, 8146, 8146)"
for s in ("train", "val"):
    assert set(dfs[s]["label"]) == set(class_names), f"{s} is missing classes"

p_tr,  y_tr  = dfs["train"]["filepath"].values, dfs["train"]["label"].values
p_val, y_val = dfs["val"]["filepath"].values,   dfs["val"]["label"].values

print(f"train / val / test : {sizes[0]:,} / {sizes[1]:,} / {sizes[2]:,}   |   {n_classes} classes")
print("test split: row count and MD5 only - it is never loaded as a dataset in this notebook.")

## 3. Raw images on the runtime disk

**Why run it:** the split CSVs store absolute paths; the pixels themselves have to be present locally.
Downloads only if `PV_DIR` is empty, so a re-run in the same session costs nothing.

**How it fits:** the last check catches the failure mode that actually bites — a split frozen under one home
directory, replayed on a runtime where those paths no longer resolve. The fix is the mount, never a rebuild.

In [ ]:
# Download PlantVillage only if it isn't already on this runtime, then check the split's paths resolve
if not PV_DIR.exists() or not any(PV_DIR.iterdir()):
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        from kaggle.api.kaggle_api_extended import KaggleApi
    PV_DIR.mkdir(parents=True, exist_ok=True)
    api = KaggleApi(); api.authenticate()
    api.dataset_download_files("abdallahalidev/plantvillage-dataset",
                               path=str(PV_DIR), unzip=True, quiet=False)
    print("Downloaded to", PV_DIR)
else:
    print("Already present:", PV_DIR)

sample  = list(p_tr[:50]) + list(p_val[:50])
missing = [p for p in sample if not Path(p).exists()]
assert not missing, (
    "The absolute paths in the split CSVs do not resolve on this runtime, e.g.\n  " + str(missing[0]) +
    "\nThe split was frozen under a different home directory. Fix the download location or the mount - "
    "do NOT rebuild the split, that would change the fingerprint.")
print(f"path check: {len(sample)} sampled files from the split all resolve on this runtime.")

## 4. The `tf.data` pipelines

**Why run it:** the model consumes batched tensors, not file paths. Images are decoded, resized to 128×128 and
left in the **0–255** range — `Rescaling(1/255)` is the first layer *inside* the model, so training, validation
and any later app normalise identically.

**How it fits:** validation is built without shuffling, so `model.predict` comes back in the same order as
`yi_val` and the metrics line up row for row. `SMOKE_TEST` caps images per class and leaves everything else alone.

In [ ]:
# Optional smoke-test subsample, then the pipelines (pixels stay 0-255)
def cap_per_class(paths, labels, cap, seed=SEED):
    rng = random.Random(seed); by = {}
    for p, l in zip(paths, labels):
        by.setdefault(l, []).append(p)
    op, ol = [], []
    for l, ps in by.items():
        sel = ps if len(ps) <= cap else rng.sample(ps, cap)
        op += sel; ol += [l] * len(sel)
    idx = np.arange(len(op)); rng.shuffle(idx)
    op, ol = np.array(op), np.array(ol)
    return op[idx], ol[idx]

if SMOKE_TEST:
    p_tr,  y_tr  = cap_per_class(p_tr,  y_tr,  SMOKE_PER_CLASS)
    p_val, y_val = cap_per_class(p_val, y_val, max(20, SMOKE_PER_CLASS // 3))
    print(f"[SMOKE_TEST] reduced to train={len(p_tr):,}, val={len(p_val):,}")

yi_tr  = np.array([name2idx[y] for y in y_tr],  dtype="int32")
yi_val = np.array([name2idx[y] for y in y_val], dtype="int32")

AUTOTUNE = tf.data.AUTOTUNE
def make_ds(paths, ilabels, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, ilabels))
    def _load(path, lab):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)   # robust to jpg/png
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])                     # float, still 0-255
        img.set_shape([IMG_SIZE, IMG_SIZE, 3])
        return img, lab
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(min(2048, len(paths)), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(p_tr,  yi_tr,  training=True)
val_ds   = make_ds(p_val, yi_val, training=False)     # order preserved -> lines up with yi_val

xb, yb = next(iter(train_ds))
print("batch:", tuple(xb.shape), "| pixel range:", float(tf.reduce_min(xb)), "-", float(tf.reduce_max(xb)))
assert float(tf.reduce_max(xb)) > 1.5, "pixels should still be 0-255 here (Rescaling is a model layer)"
print("pipelines ready.")

## 5. Balanced class weights from the training split

**Why run it:** at ~36:1 imbalance an unweighted loss chases Tomato and Orange and gives up on the 152-image
healthy Potato class.

**How it fits:** computed from the **training labels only** and passed to `fit` — the same correction every
other model in this project uses, which is what makes the macro-F1 column comparable across Table 1.

In [ ]:
# Balanced class weights from the TRAIN split -> {class_index: weight} for model.fit
weights      = compute_class_weight(class_weight="balanced", classes=np.arange(n_classes), y=yi_tr)
class_weight = {i: float(w) for i, w in enumerate(weights)}
top = sorted(class_weight.items(), key=lambda kv: kv[1], reverse=True)[:3]
print("weight range: {:.2f} (most common class) ... {:.2f} (rarest class)".format(
      min(class_weight.values()), max(class_weight.values())))
print("highest-weighted (rarest) classes:", [(class_names[i], round(w, 1)) for i, w in top])

## 6. The architecture — unchanged

`Rescaling(1/255)` → augmentation block → four `Conv2D/BatchNorm/MaxPool` blocks (32→64→128→128) → global
average pooling → dropout 0.3 → dense 128 → softmax over 38 classes. Adam at 1e-3, sparse categorical
cross-entropy.

**Why run it:** it is deliberately small and un-tuned. That is the point — the Bayesian search in Step 3.2
rediscovered this configuration on four of five knobs.

**How it fits:** Keras augmentation layers are inactive at inference, so flip / rotation / zoom apply to
training batches only and no augmented pixel ever reaches a validation score. Keeping augmentation inside the
model also means it travels with the `.keras` file.

In [ ]:
# Small baseline CNN: Rescaling + augmentation layers + 4 conv blocks + GAP head
data_augment = models.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.05, seed=SEED),
    layers.RandomZoom(0.05, seed=SEED),
], name="augment")   # inactive at inference -> train-only

def build_scratch_cnn():
    m = models.Sequential([
        layers.Input((IMG_SIZE, IMG_SIZE, 3)),
        layers.Rescaling(1.0 / 255),          # normalise INSIDE the model
        data_augment,                         # train-only augmentation
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ], name="scratch_cnn")
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

build_scratch_cnn().summary()

## 7. Skip-guard → train → save immediately

**Why run it:** this is the cell the run book asked for. If `02_models/scratch_cnn.keras` is already on Drive it
is loaded and training is skipped. Otherwise the model trains and is written to Drive **in the same cell, before
a single metric is computed** — the previous checkpoint was lost to a runtime wipe, and evaluation is exactly
the window where that happens.

**How it fits:** the save is then verified by reading the file back off Drive and running one forward pass
through it. A `.keras` file that exists but does not reload is worse than no file at all, because the skip-guard
would trust it. Training history is cached alongside, so a later re-run can still draw the learning curves
without retraining.

In [ ]:
# Skip-guard -> train -> SAVE IMMEDIATELY (before any evaluation) -> verify the file reloads
h, train_secs, trained_now = None, float("nan"), False

if MODEL_PATH.exists() and not FORCE_RETRAIN:
    model = tf.keras.models.load_model(MODEL_PATH)
    print(f"Found {MODEL_PATH.name} on Drive -> loaded, training skipped.  (FORCE_RETRAIN=True to overwrite.)")
    if HIST_PATH.exists():
        h = json.loads(HIST_PATH.read_text())
        print(f"cached history reloaded: {len(h['loss'])} epochs")
    else:
        print("no cached history on Drive -> the learning-curve cell will be skipped")
else:
    model = build_scratch_cnn()
    cbs = [
        callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
    ]
    t0 = time.time()
    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=(2 if SMOKE_TEST else EPOCHS),
                        class_weight=class_weight, callbacks=cbs, verbose=1)
    train_secs, trained_now = time.time() - t0, True
    h = {k: [float(v) for v in vs] for k, vs in history.history.items()}

    # ---- save FIRST, evaluate afterwards ----
    model.save(MODEL_PATH)
    HIST_PATH.write_text(json.dumps(h, indent=2))
    print(f"\ntrained {len(h['loss'])} epochs in {train_secs:.0f}s"
          f" | best val_accuracy during training = {max(h['val_accuracy']):.4f}")
    print("SAVED ->", MODEL_PATH)

# ---- verify the artifact on Drive is real and readable ----
assert MODEL_PATH.exists() and MODEL_PATH.stat().st_size > 0, "the model file was not written to Drive"
_check = tf.keras.models.load_model(MODEL_PATH)
_out   = _check.predict(np.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype="float32"), verbose=0)
assert _out.shape == (1, n_classes), f"reloaded model outputs {_out.shape}, expected (1, {n_classes})"
del _check
n_params = model.count_params()
print(f"verified on Drive: {MODEL_PATH}  ({MODEL_PATH.stat().st_size/1e6:.1f} MB, {n_params:,} parameters)")

## 8. Learning curves

A quick over/under-fit read before the metrics. Skipped automatically when the model was loaded from Drive and
no cached history was found.

In [ ]:
# Accuracy and loss, train vs validation
if h is None:
    print("no history available (model loaded from Drive, no cached history) -> skipping learning curves")
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h["accuracy"], label="train"); ax[0].plot(h["val_accuracy"], label="val")
    ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h["loss"], label="train"); ax[1].plot(h["val_loss"], label="val")
    ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.suptitle("Scratch CNN - learning curves"); plt.tight_layout()
    plt.savefig(LC_PNG, dpi=150, bbox_inches="tight")
    plt.show()
    print("saved ->", LC_PNG)

## 9. Validation metrics

**Why run it:** macro-F1 is the primary metric — it weights the 152-image healthy Potato class the same as the
5,000-image Tomato classes, which plain accuracy does not.

**How it fits:** one `predict` pass supplies every number below. Validation only; the test set is untouched.

In [ ]:
# One prediction pass over the validation set, then the headline metrics
y_true = yi_val
y_prob = model.predict(val_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)
assert len(y_pred) == len(y_true), "prediction/label length mismatch - val_ds must not be shuffled"

val_mf1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
val_acc = accuracy_score(y_true, y_pred)
print(f"Scratch CNN (validation, n={len(y_true):,}):   macro-F1 = {val_mf1:.4f}   accuracy = {val_acc:.4f}")

## 10. Per-class recall and the results CSVs

Two files land in `03_results/`: a one-row summary that Step 9 can concatenate straight into
`master_results.csv`, and the **full 38-row** per-class table in canonical class order — recall, precision, F1
and support, not just the worst few.

In [ ]:
# Full per-class table (all 38 classes) + the one-row summary, both written to 03_results/
rec  = recall_score(y_true, y_pred, average=None, labels=range(n_classes), zero_division=0)
prec = precision_score(y_true, y_pred, average=None, labels=range(n_classes), zero_division=0)
f1c  = f1_score(y_true, y_pred, average=None, labels=range(n_classes), zero_division=0)

per_class = pd.DataFrame({
    "class":      class_names,
    "recall":     rec,
    "precision":  prec,
    "f1":         f1c,
    "val_images": np.bincount(y_true, minlength=n_classes),
})
per_class.to_csv(PC_CSV, index=False)          # canonical class order, all 38 rows

worst = per_class.sort_values("recall").head(5)
print("Five hardest classes (lowest recall):")
print(worst.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nFive easiest classes:")
print(per_class.sort_values("recall").tail(5).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

summary = pd.DataFrame([{
    "model":                "Scratch CNN",
    "description":          "from-scratch CNN, 4 conv blocks, 128px, unchanged Step 3.1 recipe",
    "split_id":             SPLIT_ID,
    "val_macro_f1":         round(float(val_mf1), 4),
    "val_accuracy":         round(float(val_acc), 4),
    "min_per_class_recall": round(float(per_class["recall"].min()), 4),
    "worst_class":          per_class.loc[per_class["recall"].idxmin(), "class"],
    "parameters":           int(n_params),
    "epochs_run":           (len(h["loss"]) if h else None),
    "train_seconds":        (round(train_secs) if trained_now else None),
    "img_size":             IMG_SIZE,
    "batch_size":           BATCH_SIZE,
    "class_weights":        "balanced",
    "augmentation":         "RandomFlip + RandomRotation(0.05) + RandomZoom(0.05), train-only",
    "seed":                 SEED,
    "n_train":              int(len(p_tr)),
    "n_val":                int(len(p_val)),
    "trained_this_run":     trained_now,
    "smoke_test":           SMOKE_TEST,
    "timestamp":            pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
}])
summary.to_csv(RES_CSV, index=False)

print("\n" + "-" * 70)
print(summary.T.to_string(header=False))
print("-" * 70)
print("\nwrote:", RES_CSV.name, "and", PC_CSV.name)

In [ ]:
# Full precision / recall / F1 / support breakdown, for the record
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

## 11. Confusion matrix

Row-normalised, so the diagonal reads directly as per-class recall. The surviving off-diagonal mass is what the
error-analysis section of the report is about — and, in this project, it concentrates within a crop rather than
across crops.

In [ ]:
# Row-normalised confusion matrix -> 04_figures/
cm      = confusion_matrix(y_true, y_pred, labels=range(n_classes)).astype(float)
row_sum = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row_sum, out=np.zeros_like(cm), where=row_sum > 0)

plt.figure(figsize=(14, 12))
sns.heatmap(cm_norm, cmap="viridis", vmin=0, vmax=1, square=True,
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={"shrink": 0.7, "label": "fraction of true class"})
plt.xticks(rotation=90, fontsize=7); plt.yticks(rotation=0, fontsize=7)
plt.xlabel("predicted"); plt.ylabel("true (diagonal = recall)")
plt.title(f"Scratch CNN - row-normalised confusion matrix (validation, macro-F1 {val_mf1:.3f})")
plt.tight_layout()
plt.savefig(CM_PNG, dpi=150, bbox_inches="tight")
plt.show()
print("saved ->", CM_PNG)

# The confusion pairs that actually survive, for the error-analysis section
off = cm_norm.copy(); np.fill_diagonal(off, 0.0)
pairs = [(class_names[i], class_names[j], off[i, j], int(cm[i, j]))
         for i, j in zip(*np.unravel_index(np.argsort(off, axis=None)[::-1][:8], off.shape))]
print("\nTop confusions (true -> predicted, share of the true class):")
for t, p, frac, cnt in pairs:
    if frac > 0:
        print(f"  {frac:5.1%}  ({cnt:>3} imgs)   {t}  ->  {p}")

## 12. What was written

The run's paper trail, in one place. Everything below is on Drive and survives a runtime wipe.

In [ ]:
# Manifest of everything this notebook wrote to Drive
print("SPLIT_ID :", SPLIT_ID)
print("val macro-F1 / accuracy : {:.4f} / {:.4f}".format(val_mf1, val_acc))
print()
for p in (MODEL_PATH, HIST_PATH, RES_CSV, PC_CSV, LC_PNG, CM_PNG):
    if p.exists():
        print(f"  {p.stat().st_size/1e6:8.2f} MB   {p.relative_to(D2)}")
    else:
        print(f"  {'-':>8}      {p.relative_to(D2)}   (not written)")
if SMOKE_TEST:
    print("\n*** SMOKE_TEST run - these are _SMOKE artifacts. Set SMOKE_TEST=False for the real run. ***")

---
### Reading the result

Record whatever macro-F1 this run produced, next to the `SPLIT_ID`, and use **that** number in Table 1. The
0.979 on record came from a different session; GPU training is not bit-deterministic and a spread of roughly
±0.02 is normal here. A number inside that band is the same model, not a regression — a number outside it means
something in the recipe or the split moved, and the fingerprint is the first thing to check.

**What this checkpoint is for.** Two things downstream depend on the file rather than the number:

- **Step 5 — Grad-CAM.** The paired attention comparison against DenseNet-121 needs both models live in one
  session. That is why the save happens before evaluation.
- **Step 6 — the background-shortcut probe.** Scores this checkpoint and DenseNet-121 on the same validation
  files in the colour and segmented renderings.

The test set has not been touched. It is opened once, on the final model, in `12_final_test_evaluation.ipynb`.